# 00 · Setup and self-check (SageMaker Studio, GPU)

Run this once per instance. Recommended: **ml.g5.xlarge** (A10G) or **ml.g6.xlarge** (L4), image **PyTorch 2.x GPU**.

1. installs the requirements
2. runs the unit tests (includes the GPU crop check that the CPU-only CI skips)
3. loads every model once: RF-DETR, RTMPose, PARSeq, DINOv2, and NVDEC decoding
4. re-measures the blog's profiling surprise: building pose crops on the CPU vs the GPU

In [ ]:
import os, sys, json, time
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))
os.chdir(REPO)
print("working in", REPO)

In [ ]:
!nvidia-smi
!pip install -q -r requirements.txt
# TensorRT for notebook 03 (match the image's CUDA major version)
!pip install -q tensorrt-cu12 || echo "TensorRT install failed: notebook 03 needs it, the rest does not"

In [ ]:
import torch, cv2, numpy as np
print("torch", torch.__version__, "cuda", torch.version.cuda, "gpu", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "pick a GPU instance"

## Unit tests
The torch crop test runs here (it is skipped on machines without PyTorch).

In [ ]:
!python -m pytest -q tests

## Models load and run

In [ ]:
from vb.config import load_config
from vb.detect import Detector
cfg = load_config("configs/default.yaml")
det = Detector(cfg.detector, "cuda")
img = np.full((720, 1280, 3), 120, np.uint8)
cv2.circle(img, (640, 300), 12, (0, 220, 255), -1)          # a ball-like blob, players come from your clip in 01
x = torch.from_numpy(img[..., ::-1].copy()).permute(2, 0, 1).float().div(255).cuda()[None]
d = det(x)[0]
print("RF-DETR ok:", len(d.player_boxes), "players,", len(d.ball_xy), "balls on a synthetic frame")

In [ ]:
from vb.pose import PoseEstimator
pose = PoseEstimator(cfg.pose, "cuda")
frame = torch.from_numpy(img).permute(2, 0, 1).float().cuda()     # BGR 0..255
kp, sc = pose(frame, img, np.float32([[600, 200, 680, 420]]))
print("RTMPose ok:", kp.shape, sc.shape)

In [ ]:
from vb.identity import JerseyReader, AppearanceEmbedder, letterbox_text
reader = JerseyReader(cfg.identity.parseq_hub, cfg.identity.parseq_model, "cuda")
shirt = np.full((64, 64, 3), (40, 40, 200), np.uint8)
cv2.putText(shirt, "12", (6, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.6, (255, 255, 255), 4)
print("PARSeq read:", reader([letterbox_text(shirt)]))   # expect ('12', high confidence)
emb = AppearanceEmbedder(cfg.identity.dino_model, "cuda")
print("DINOv2 ok:", emb([shirt]).shape)

In [ ]:
# NVDEC: put the path or s3:// URI of any clip here
from vb.io import fetch, VideoSource
clip = fetch(cfg.video.source, cfg.video.local_cache) if cfg.video.source.startswith("s3://") else None
if clip:
    src = VideoSource(clip, "auto", "cuda")
    print("decoder:", src.decoder, "fps", src.fps, "size", src.size, "frames", src.n_frames)
else:
    print("set video.source in configs/default.yaml to test NVDEC on your clip")

## The profiling surprise, re-measured
RTMPose is top-down: every person box becomes a 256×192 crop. Building those crops one by one on the CPU
(what most code does) against one batched `affine_grid` + `grid_sample` on the GPU, for 22 people:

In [ ]:
from vb.pose import boxes_to_center_scale, batched_crops_cpu, batched_crops_torch
H, W = 1080, 1920
frame_np = (np.random.rand(H, W, 3) * 255).astype(np.float32)
frame_gpu = torch.from_numpy(frame_np).permute(2, 0, 1).contiguous().cuda()
rng = np.random.default_rng(0)
xy = rng.uniform([100, 100], [W - 200, H - 400], (22, 2))
boxes = np.float32(np.c_[xy, xy + rng.uniform([40, 120], [90, 300], (22, 2))])
c, s = boxes_to_center_scale(boxes)
def bench(fn, n=50):
    fn(); torch.cuda.synchronize(); t = time.perf_counter()
    for _ in range(n): fn()
    torch.cuda.synchronize(); return 1000 * (time.perf_counter() - t) / n
cpu_ms = bench(lambda: batched_crops_cpu(frame_np, c, s), 20)
gpu_ms = bench(lambda: batched_crops_torch(frame_gpu, c, s))
diff = np.abs(batched_crops_torch(frame_gpu, c, s).cpu().numpy() - batched_crops_cpu(frame_np, c, s))[:, :, 2:-2, 2:-2].max()
print(f"22 crops: CPU {cpu_ms:.1f} ms/frame, GPU {gpu_ms:.2f} ms/frame, max diff {diff:.4f} (std units)")